In [3]:
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
import scanpy as sc
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib_scalebar.scalebar import ScaleBar

# Define oncogene and suppressor markers, including NF1
oncogene_markers = ["KRAS", "NRAS", "BRAF"]
suppressor_markers = ["SPRED1", "PTEN", "NF1"]

# Define directory for .h5ad files
directory = '/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples'
file_paths = glob.glob(os.path.join(directory, '*.h5ad'))

# Loop over each file path
for file_path in file_paths:
    print(f"Processing file: {file_path}")
    try:
        # Read the adata file
        adata = sc.read_h5ad(file_path)

        # Ensure unique variable names
        adata.var_names_make_unique()

        # Extract histological image and scale factor
        library_id = list(adata.uns['spatial'].keys())[0]  # Use the first library
        scale_factor = adata.uns['spatial'][library_id]['scalefactors']['tissue_hires_scalef']  # Scale factor for spatial coords

        # Initialize DataFrames to store scores
        cells_scores = pd.DataFrame(index=adata.obs.index)
        cells_scores['Oncogene Score'] = 0
        cells_scores['Suppressor Score'] = 0

        # Calculate Oncogene Score
        for gene in oncogene_markers:
            if gene in adata.var_names:
                gene_expression = adata[:, gene].X.toarray().ravel()
                if np.std(gene_expression) != 0:
                    normalized_expression = (gene_expression - np.mean(gene_expression)) / np.std(gene_expression)
                    cells_scores['Oncogene Score'] += normalized_expression
            else:
                print(f"Gene {gene} not found in dataset. Skipping...")

        # Calculate Suppressor Score
        for gene in suppressor_markers:
            if gene in adata.var_names:
                gene_expression = adata[:, gene].X.toarray().ravel()
                if np.std(gene_expression) != 0:
                    normalized_expression = (gene_expression - np.mean(gene_expression)) / np.std(gene_expression)
                    cells_scores['Suppressor Score'] += normalized_expression
            else:
                print(f"Gene {gene} not found in dataset. Skipping...")

        # Skip if no markers contribute to the scores
        if cells_scores['Oncogene Score'].sum() == 0 and cells_scores['Suppressor Score'].sum() == 0:
            print(f"No valid markers found for {os.path.basename(file_path)}. Skipping file...")
            continue

        # Normalize scores between 0 and 1 with clipping to reduce black regions
        oncogene_norm = Normalize(vmin=0, vmax=np.percentile(cells_scores['Oncogene Score'], 95), clip=True)
        suppressor_norm = Normalize(vmin=0, vmax=np.percentile(cells_scores['Suppressor Score'], 95), clip=True)
        cells_scores['Oncogene Score'] = oncogene_norm(cells_scores['Oncogene Score'])
        cells_scores['Suppressor Score'] = suppressor_norm(cells_scores['Suppressor Score'])

        # Create RGB blended colors
        rgb_colors = np.zeros((len(cells_scores), 3))
        rgb_colors[:, 0] = cells_scores['Oncogene Score']  # Red channel
        rgb_colors[:, 2] = cells_scores['Suppressor Score']  # Blue channel

        # Scale spatial coordinates to match the image
        spatial_coords = adata.obsm['spatial'] * scale_factor

        # Plotting
        # Scatter plot of blended colors
        plt.figure(figsize=(12, 10))
        ax = plt.gca()  # Get the current axes
        scatter = ax.scatter(
            spatial_coords[:, 0],
            spatial_coords[:, 1],
            c=rgb_colors,
            alpha=0.8,  # Reduce transparency for better visibility
            s=20       # Adjust dot size
        )
        
        # Add color bars explicitly tied to the plot
        oncogene_colorbar = plt.colorbar(
            plt.cm.ScalarMappable(norm=oncogene_norm, cmap="Reds"),
            ax=ax,  # Explicitly assign the axis
            label="Oncogene Score (Red)", shrink=0.8, pad=0.02
        )
        suppressor_colorbar = plt.colorbar(
            plt.cm.ScalarMappable(norm=suppressor_norm, cmap="Blues"),
            ax=ax,  # Explicitly assign the axis
            label="Suppressor Score (Blue)", shrink=0.8, pad=0.12
        )
        
        # Add scale bar (already included, so no change here)
        scalebar = ScaleBar(scale_factor, units="µm", location="lower right", color="black", box_color="white", box_alpha=0.5)
        ax.add_artist(scalebar)
        
        # Formatting
        ax.axis("off")
        plt.title(f"Blended Oncogene and Suppressor Activity (NF1 Included)\n{os.path.basename(file_path)}", fontsize=14)
        
        # Save the plot
        output_path = f"{os.path.splitext(file_path)[0]}_blended_oncogenes_suppressors_refined.png"
        plt.savefig(output_path, bbox_inches="tight", dpi=300)
        plt.close()
        print(f"Saved plot to {output_path}")
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        continue

Processing file: /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/51_A10-38_0_adata.h5ad


/opt/anaconda3/lib/python3.12/site-packages/matplotlib_scalebar/scalebar.py:457: UserWarning: Drawing scalebar on axes with unequal aspect ratio; either call ax.set_aspect(1) or suppress the warning with rotation='horizontal-only'.
  warnings.warn(


Saved plot to /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/51_A10-38_0_adata_blended_oncogenes_suppressors_refined.png
Processing file: /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/128_B9-102_B12_1_adata.h5ad


/opt/anaconda3/lib/python3.12/site-packages/matplotlib_scalebar/scalebar.py:457: UserWarning: Drawing scalebar on axes with unequal aspect ratio; either call ax.set_aspect(1) or suppress the warning with rotation='horizontal-only'.
  warnings.warn(


Saved plot to /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/128_B9-102_B12_1_adata_blended_oncogenes_suppressors_refined.png
Processing file: /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/80_A12-96_A8_0_adata.h5ad


...

Saved plot to /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/106_A4-60_B3_1_adata_blended_oncogenes_suppressors_refined.png
Processing file: /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/46_A31-111_B6_0_adata.h5ad


/opt/anaconda3/lib/python3.12/site-packages/matplotlib_scalebar/scalebar.py:457: UserWarning: Drawing scalebar on axes with unequal aspect ratio; either call ax.set_aspect(1) or suppress the warning with rotation='horizontal-only'.
  warnings.warn(


Saved plot to /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/46_A31-111_B6_0_adata_blended_oncogenes_suppressors_refined.png
Processing file: /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/97_A7-81_A11_1_adata.h5ad


/opt/anaconda3/lib/python3.12/site-packages/matplotlib_scalebar/scalebar.py:457: UserWarning: Drawing scalebar on axes with unequal aspect ratio; either call ax.set_aspect(1) or suppress the warning with rotation='horizontal-only'.
  warnings.warn(


Saved plot to /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples/97_A7-81_A11_1_adata_blended_oncogenes_suppressors_refined.png
